# 验证 AgiBot：源 → L1 → L2

对照[统一格式字段及命名标准(LeRobot v3)](https://zcnce50wan15.feishu.cn/wiki/Jitbwmov2iJMQpkpy6Zc0TPMnJf)：

1. **源 ↔ L1**：数值零变换（joint / ee_pose / gripper_mm）
2. **L1 → L2**：open01 / pose9 / delta_chunk / pad；相机仅 head + wrist_left/right；**无内外参**
3. **官方可读**：`LeRobotDataset` / `open_dataset`

先跑：
```bash
cd phi_rds && python examples/agibot_l0_l1_l2.py --max-episodes 1
jupyter notebook examples/verify_agibot_l1_l2.ipynb
```

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import h5py
import numpy as np
import pyarrow.parquet as pq
from scipy.spatial.transform import Rotation as R

HERE = Path.cwd()
if (HERE / "src" / "phi_rds").is_dir():
    REPO = HERE.parent
    sys.path.insert(0, str(HERE / "src"))
elif (HERE.parent / "src" / "phi_rds").is_dir():
    REPO = HERE.parent.parent
    sys.path.insert(0, str(HERE.parent / "src"))
else:
    REPO = HERE.parents[1] if HERE.name == "examples" else HERE

SRC = REPO / "data/agibot_demo/sample_dataset"
L1 = REPO / "data/output/agibot_v3_raw/demo/g1"
L2 = REPO / "data/output/agibot_v3_sem/demo/g1"
TASK_ID, EP_ID = 355, 662854
GRIP_OPEN_MM = 80.0

assert (L1 / "meta/info.json").is_file(), f"先跑 pipeline 生成 L1: {L1}"
assert (L2 / "meta/info.json").is_file(), f"先跑 pipeline 生成 L2: {L2}"
print("SRC", SRC)
print("L1 ", L1)
print("L2 ", L2)

SRC /Users/rookie/Desktop/EAI/phi_data/data/agibot_demo/sample_dataset
L1  /Users/rookie/Desktop/EAI/phi_data/data/output/agibot_v3_raw/demo/g1
L2  /Users/rookie/Desktop/EAI/phi_data/data/output/agibot_v3_sem/demo/g1


## 工具

In [3]:
def pack_ee(pos, ori):
    """(N,2,3)+(N,2,4) → (N,14) left xyz+quat | right xyz+quat"""
    pos = np.asarray(pos, dtype=np.float64).reshape(-1, 2, 3)
    ori = np.asarray(ori, dtype=np.float64).reshape(-1, 2, 4)
    return np.concatenate([pos[:, 0], ori[:, 0], pos[:, 1], ori[:, 1]], axis=1)


def load_ep0_table(root: Path):
    rows = pq.read_table(root / "meta/episodes/chunk-000/file-000.parquet").to_pylist()
    m = rows[0]
    table = pq.read_table(
        root / "data" / f"chunk-{int(m['data/chunk_index']):03d}"
        / f"file-{int(m['data/file_index']):03d}.parquet"
    )
    mask = np.asarray(table.column("episode_index").to_pylist()) == 0
    return table, mask, m


def col(table, mask, name):
    vals = np.asarray(table.column(name).to_pylist())[mask]
    return np.stack(vals) if getattr(vals, "dtype", None) == object else np.asarray(vals)


def max_abs(a, b):
    return float(np.max(np.abs(np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64))))


def check(name, err, tol=0.0):
    ok = err <= tol
    mark = "OK" if ok else "FAIL"
    print(f"  [{mark}] {name:<40} max|Δ|={err:.3e}  (tol={tol})")
    return ok

## 1. 读源 h5

In [4]:
h5_path = SRC / "proprio_stats" / str(TASK_ID) / str(EP_ID) / "proprio_stats.h5"
with h5py.File(h5_path, "r", locking=False) as f:
    st, act = f["state"], f["action"]
    src = {
        "joint": np.asarray(st["joint"]["position"], dtype=np.float64),
        "grip": np.asarray(st["effector"]["position"], dtype=np.float64),
        "ee_pos": np.asarray(st["end"]["position"], dtype=np.float64),
        "ee_ori": np.asarray(st["end"]["orientation"], dtype=np.float64),
        "waist": np.asarray(st["waist"]["position"], dtype=np.float64),
        "head": np.asarray(st["head"]["position"], dtype=np.float64),
        "a_joint": np.asarray(act["joint"]["position"], dtype=np.float64),
        "a_grip": np.asarray(act["effector"]["position"], dtype=np.float64),
        "a_ee_pos": np.asarray(act["end"]["position"], dtype=np.float64),
        "a_ee_ori": np.asarray(act["end"]["orientation"], dtype=np.float64),
        "ts": np.asarray(f["timestamp"], dtype=np.int64),
    }

src["ee"] = pack_ee(src["ee_pos"], src["ee_ori"])
src["a_ee"] = pack_ee(src["a_ee_pos"], src["a_ee_ori"])
n = len(src["joint"])
print(f"frames={n}")
print("ee[0]", src["ee"][0])
print("grip_mm[0]", src["grip"][0])

frames=2418
ee[0] [ 0.60263094  0.29837693  0.56728469 -0.07100343  0.74494317 -0.66139842
  0.05069837  0.60320899 -0.29776498  0.56315803 -0.73955087  0.06565339
 -0.04971038  0.66804417]
grip_mm[0] [34.93333333 34.66666667]


## 2. 源 ↔ L1（必须 max|Δ|=0）

In [5]:
info1 = json.loads((L1 / "meta/info.json").read_text())
sem = json.loads((L1 / "meta/field_semantics.json").read_text())
t1, m1, _ = load_ep0_table(L1)

print("L1 codebase", info1["codebase_version"], "phi_level", info1.get("phi_level"))
print("ee_pose names", info1["features"]["observation.state.arm.ee_pose"]["names"])
print("quat_order", sem["features"]["observation.state.arm.ee_pose"]["quaternion_order"])
print()

ok = True
ok &= check("state.joint", max_abs(col(t1, m1, "observation.state.arm.joint"), src["joint"]))
ok &= check("state.ee_pose", max_abs(col(t1, m1, "observation.state.arm.ee_pose"), src["ee"]))
ok &= check("state.gripper_mm", max_abs(col(t1, m1, "observation.state.arm.gripper"), src["grip"]))
ok &= check("state.torso", max_abs(col(t1, m1, "observation.state.torso.joint"), src["waist"]))
ok &= check("state.head", max_abs(col(t1, m1, "observation.state.head.joint"), src["head"]))
ok &= check("action.joint", max_abs(col(t1, m1, "action.arm.joint"), src["a_joint"]))
ok &= check("action.ee_pose", max_abs(col(t1, m1, "action.arm.ee_pose"), src["a_ee"]))
ok &= check("action.gripper_closed", max_abs(col(t1, m1, "action.arm.gripper"), src["a_grip"]))
ok &= check("timestamp_ns", max_abs(col(t1, m1, "observation.timestamp_ns").ravel(), src["ts"]))

videos = sorted((L1 / "videos").rglob("*.mp4"))
print(f"  videos={len(videos)}")
assert len(videos) == 8, videos
assert ok, "L1 lossless check failed"
print("\nL1 PASS")

L1 codebase v3.0 phi_level L1
ee_pose names ['left_x', 'left_y', 'left_z', 'left_qx', 'left_qy', 'left_qz', 'left_qw', 'right_x', 'right_y', 'right_z', 'right_qx', 'right_qy', 'right_qz', 'right_qw']
quat_order xyzw

  [OK] state.joint                              max|Δ|=0.000e+00  (tol=0.0)
  [OK] state.ee_pose                            max|Δ|=0.000e+00  (tol=0.0)
  [OK] state.gripper_mm                         max|Δ|=0.000e+00  (tol=0.0)
  [OK] state.torso                              max|Δ|=0.000e+00  (tol=0.0)
  [OK] state.head                               max|Δ|=0.000e+00  (tol=0.0)
  [OK] action.joint                             max|Δ|=0.000e+00  (tol=0.0)
  [OK] action.ee_pose                           max|Δ|=0.000e+00  (tol=0.0)
  [OK] action.gripper_closed                    max|Δ|=0.000e+00  (tol=0.0)
  [OK] timestamp_ns                             max|Δ|=0.000e+00  (tol=0.0)
  videos=8

L1 PASS


## 3. L1 → L2（飞书标准：delta_chunk / pad / 三路 RGB / 无内外参）

In [6]:
info2 = json.loads((L2 / "meta/info.json").read_text())
t2, m2, _ = load_ep0_table(L2)
feats = info2["features"]

print("L2 codebase", info2["codebase_version"], "phi_level", info2.get("phi_level"),
      "std", info2.get("phi_standard_version"))
print("features", sorted(k for k, v in feats.items() if v.get("dtype") != "video"))
print()

ok = True

# --- gripper: mm → open01 ---
g_l1 = col(t1, m1, "observation.state.arm.gripper")
g_l2 = col(t2, m2, "observation.state.arm.gripper")
ok &= check("state.gripper open01", max_abs(g_l2, np.clip(g_l1 / GRIP_OPEN_MM, 0, 1)), tol=1e-6)
ag_l1 = col(t1, m1, "action.arm.gripper")
ag_l2 = col(t2, m2, "action.arm.gripper")
ok &= check("action.gripper open01", max_abs(ag_l2, np.clip(1.0 - ag_l1, 0, 1)), tol=1e-6)

# --- joint 透传（float32 容差）---
ok &= check(
    "state.joint L1→L2",
    max_abs(col(t2, m2, "observation.state.arm.joint"), col(t1, m1, "observation.state.arm.joint")),
    tol=1e-5,
)

# --- ee_pose (xyz+quat) → tcp_pose (pose9×2=18) ---
from phi_rds import encode_pose

ee = col(t1, m1, "observation.state.arm.ee_pose").reshape(-1, 14)
tcp = col(t2, m2, "observation.state.arm.tcp_pose")
assert tcp.shape[1] == 18, tcp.shape

def pose9_from_xyz_quat(block7):
    T = np.tile(np.eye(4), (len(block7), 1, 1))
    T[:, :3, 3] = block7[:, :3]
    T[:, :3, :3] = R.from_quat(block7[:, 3:7]).as_matrix()
    return encode_pose(T, "pose9")

tcp_expect = np.concatenate(
    [pose9_from_xyz_quat(ee[:, :7]), pose9_from_xyz_quat(ee[:, 7:14])], axis=1,
)
ok &= check("state.tcp_pose from ee_pose", max_abs(tcp, tcp_expect), tol=1e-5)

# --- 标准字段：delta_chunk + pad ---
assert feats["observation.state.arm.tcp_pose.delta_chunk"]["shape"] == [10, 18]
assert feats["action.arm.tcp_pose.delta_chunk"]["shape"] == [64, 18]
assert feats["observation.state.torso.joint"]["pad"] == [0, 2]
assert feats["observation.state.head.joint"]["pad"] == [2]
dc = col(t2, m2, "observation.state.arm.tcp_pose.delta_chunk")
assert dc.shape[1:] == (10, 18), dc.shape
torso = col(t2, m2, "observation.state.torso.joint")
head = col(t2, m2, "observation.state.head.joint")
ok &= check("torso pad slots", max_abs(torso[:, [0, 2]], 0))
ok &= check("torso pitch/lift", max_abs(torso[:, [1, 3]], col(t1, m1, "observation.state.torso.joint")), tol=1e-5)
ok &= check("head pad roll", max_abs(head[:, 2], 0))
ok &= check("head yaw/pitch", max_abs(head[:, :2], col(t1, m1, "observation.state.head.joint")), tol=1e-5)
# torso/head.pose：URDF FK（与 agibot_pad_demo 对齐）；仅 base 全 pad
assert feats["observation.state.head.pose"].get("pad") in (None, [])
assert feats["observation.state.torso.pose"].get("pad") in (None, [])
assert feats["observation.state.base.pose"].get("pad") == list(range(9))
hp = col(t2, m2, "observation.state.head.pose")
ok &= check("head.pose nonzero (FK)", 0.0 if float(np.max(np.abs(hp))) > 0.1 else 1.0)
print("  [OK] delta_chunk + pad schema + FK poses")

# --- 相机：仅 head + wrists；无 cameras 内外参 ---
vids2 = sorted(p.name for p in (L2 / "videos").iterdir() if p.is_dir())
want = {"observation.images.head", "observation.images.wrist_left", "observation.images.wrist_right"}
cam_ok = set(vids2) == want
episodes = pq.read_table(L2 / "meta/episodes/chunk-000/file-000.parquet")
no_calib = "cameras" not in episodes.column_names
print("  L2 video keys", vids2)
print(f"  [{'OK' if cam_ok else 'FAIL'}] camera subset (head + wrists only)")
print(f"  [{'OK' if no_calib else 'FAIL'}] no episode cameras/intrinsics column")
ok &= cam_ok and no_calib

assert ok, "L2 semantic check failed"
print("\nL2 PASS")

L2 codebase v3.0 phi_level L2 std 2026.04
features ['action.arm.gripper', 'action.arm.joint', 'action.arm.tcp_pose', 'episode_index', 'frame_index', 'index', 'observation.state.arm.gripper', 'observation.state.arm.joint', 'observation.state.arm.tcp_pose', 'task_index', 'timestamp']

  [OK] state.gripper open01                     max|Δ|=2.909e-08  (tol=1e-06)
  [OK] action.gripper open01                    max|Δ|=2.623e-08  (tol=1e-06)
  [FAIL] state.joint L1→L2                        max|Δ|=5.960e-08  (tol=0.0)
  [OK] state.tcp_pose from ee_pose              max|Δ|=2.980e-08  (tol=1e-05)
  L2 video keys ['observation.images.head', 'observation.images.wrist_left', 'observation.images.wrist_right']
  [FAIL] camera subset


AssertionError: L2 semantic check failed

## 4. 官方 / SDK 读回

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from phi_rds import open_dataset, encode_pose

for label, root in [("L1", L1), ("L2", L2)]:
    ds = LeRobotDataset(repo_id=f"local/{label}", root=root, download_videos=False)
    item = ds[0]
    print(f"{label}: frames={len(ds)} eps={ds.num_episodes} sample_keys={len(item)} "
          f"has_image={any('images' in k for k in item)}")

ds2 = open_dataset(L2)
ep = ds2[0]
Tl = ep.state.arm_left.ee.as_matrix()
print(f"SDK L2 uid={ep.uid} eeL={Tl.shape} pose9={encode_pose(Tl, 'pose9').shape} "
      f"grip0={float(ep.state.arm_left.gripper.numpy().ravel()[0]):.4f}")
print("OK")

## 5. 一览：夹爪 / 末端首帧

In [ ]:
import pandas as pd

g_l1 = col(t1, m1, "observation.state.arm.gripper")[0]
g_l2 = col(t2, m2, "observation.state.arm.gripper")[0]
ee0 = col(t1, m1, "observation.state.arm.ee_pose")[0]
tcp0 = col(t2, m2, "observation.state.arm.tcp_pose")[0]

display(pd.DataFrame([
    {"side": "left", "L1_mm": g_l1[0], "L2_open01": g_l2[0], "mm/80": g_l1[0] / GRIP_OPEN_MM},
    {"side": "right", "L1_mm": g_l1[1], "L2_open01": g_l2[1], "mm/80": g_l1[1] / GRIP_OPEN_MM},
]))

names = info1["features"]["observation.state.arm.ee_pose"]["names"]
display(pd.DataFrame({"name": names, "L1_ee_pose[0]": ee0}))
print("L2 tcp_pose[0] left_pose9", tcp0[:9])
print("L2 tcp_pose[0] right_pose9", tcp0[9:])